In [0]:
#Query plan inspection (physical + logical)
spark.sql("""
  SELECT user_id, product_id, event_ts, price
  FROM silver.events
  WHERE event_type = 'purchase'
""").explain(True)


In [0]:
%sql
EXPLAIN FORMATTED
SELECT user_id, product_id, event_ts, price
FROM silver.events
WHERE event_type = 'purchase';


In [0]:

%sql
-- Create partitioned Silver table variant
CREATE TABLE IF NOT EXISTS silver.events_part
USING DELTA
PARTITIONED BY (event_date, event_type)
AS
SELECT
  *
FROM silver.events;


In [0]:
# Compaction fallback
from pyspark.sql import functions as F

df = spark.table("silver.events_part")

# Tune this based on size; goal is fewer, larger files without going extreme
target_partitions = 64

(df.repartition(target_partitions, F.col("event_date"), F.col("event_type"))
  .write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("silver.events_part"))


In [0]:
# ZORDER
try:
    spark.sql("OPTIMIZE silver.events_part ZORDER BY (user_id, product_id)")
except Exception as e:
    print("OPTIMIZE/ZORDER not available in this environment:", e)


In [0]:
# Benchmark harness (simple, repeatable)
import time

def bench(name: str, sql: str, n: int = 3):
    times = []
    for _ in range(n):
        t0 = time.time()
        spark.sql(sql).count()
        times.append(time.time() - t0)
    print(f"{name}: " + ", ".join([f"{t:.2f}s" for t in times]) + f" | best={min(times):.2f}s")

bench("purchase_scan (silver.events)",
      "SELECT * FROM silver.events WHERE event_type='purchase'")

bench("purchase_scan (silver.events_part)",
      "SELECT * FROM silver.events_part WHERE event_type='purchase'")

bench("user_lookup (events_part)",
      "SELECT * FROM silver.events_part WHERE user_id IS NOT NULL LIMIT 10000")
